In [ ]:
import os
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt

# Input and output folders
input_dir = 'ALL PREPROCESSED'  # <- put your EEG CSV files here
output_dir = './dwt_plots/'
os.makedirs(output_dir, exist_ok=True)

# EEG Channels from EMOTIV EPOC+ (excluding reference P3/P4)
EEG_CHANNELS = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1',
                'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']

# DWT config
wavelet = 'db2'
level = 4

# Process each subject and game file
for subject in range(1, 29):
    for game in range(1, 5):
        filename = f'S{subject:02d}G{game}AllChannels.csv'
        filepath = os.path.join(input_dir, filename)
        if not os.path.exists(filepath):
            print(f'[!] Skipping missing file: {filename}')
            continue

        df = pd.read_csv(filepath)

        for ch in EEG_CHANNELS:
            if ch not in df.columns:
                print(f'[!] Channel {ch} missing in {filename}')
                continue

            signal = df[ch].values
            coeffs = pywt.wavedec(signal, wavelet, level=level)
            labels = [f'D{i}' for i in range(1, level + 1)] + [f'A{level}']

            # Plot DWT
            fig, axs = plt.subplots(len(coeffs), 1, figsize=(12, 8), sharex=True)
            fig.suptitle(f'DWT - {filename} - {ch}', fontsize=14)

            for i, (coeff, label) in enumerate(zip(coeffs, labels)):
                axs[i].plot(coeff, color='black')
                axs[i].set_ylabel(label)
                axs[i].grid(True)

            axs[-1].set_xlabel('Samples')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])

            # Save the plot
            save_path = os.path.join(output_dir, f'{filename[:-4]}_{ch}_DWT.png')
            plt.savefig(save_path)
            plt.close()
            print(f'[+] Saved: {save_path}')


[+] Saved: ./dwt_plots/S01G1AllChannels_AF3_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_F7_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_F3_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_FC5_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_T7_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_P7_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_O1_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_O2_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_P8_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_T8_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_FC6_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_F4_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_F8_DWT.png
[+] Saved: ./dwt_plots/S01G1AllChannels_AF4_DWT.png
[+] Saved: ./dwt_plots/S01G2AllChannels_AF3_DWT.png
[+] Saved: ./dwt_plots/S01G2AllChannels_F7_DWT.png
[+] Saved: ./dwt_plots/S01G2AllChannels_F3_DWT.png
[+] Saved: ./dwt_plots/S01G2AllChannels_FC5_DWT.png
[+] Saved: ./dwt_plots/S01G2AllChannels_T7_DWT.png
[+] Saved: ./dwt_plots/S0

In [2]:
import os
import numpy as np
import pandas as pd
import pywt
from scipy.stats import entropy
from antropy import sample_entropy, spectral_entropy, detrended_fluctuation, hjorth_params

# EEG Channels
EEG_CHANNELS = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1',
                'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']

# Feature extraction from D1 to D4
def extract_features_from_details(signal, wavelet='db2'):
    coeffs = pywt.wavedec(signal, wavelet, level=4)
    details = coeffs[1:5]  # D1 to D4
    features = []
    
    for c in details:
        f = [
            np.mean(c),
            np.std(c),
            np.var(c),
            np.sum(np.diff(np.sign(c)) != 0),  # ZCR
            np.sum(c ** 2),                    # Energy
            entropy(np.histogram(c, bins=10)[0] + 1),  # Shannon entropy
            np.sum(np.log(np.square(c) + 1e-12)),      # Log energy
            sample_entropy(c),
            spectral_entropy(c, sf=128, method='fft'),
            detrended_fluctuation(c)
        ]
        try:
            f += list(hjorth_params(c))
        except:
            f += [0, 0, 0]
        features.extend(f)
    return features

# Setup paths
input_dir = 'ALL PREPROCESSED'  # Path to your EEG CSVs
output_dir = './channel_features_dwt/'
os.makedirs(output_dir, exist_ok=True)

# Prepare writers for each channel
channel_files = {
    ch: open(os.path.join(output_dir, f'{ch}_features.csv'), 'w') for ch in EEG_CHANNELS
}
feature_names = ['Mean', 'Std', 'Var', 'ZCR', 'Energy', 'Entropy',
                 'LogEntropy', 'SampleEntropy', 'SpecEntropy', 'DFA',
                 'Hjorth_Act', 'Hjorth_Mob', 'Hjorth_Comp']
header = ','.join(['Subject', 'Game'] + [f'{fn}_D{i}' for i in range(1, 5) for fn in feature_names])
for f in channel_files.values():
    f.write(header + '\n')

# Loop through all files
for subject in range(1, 29):
    for game in range(1, 5):
        filename = f'S{subject:02d}G{game}AllChannels.csv'
        filepath = os.path.join(input_dir, filename)
        if not os.path.exists(filepath):
            continue

        df = pd.read_csv(filepath)
        for ch in EEG_CHANNELS:
            if ch not in df.columns:
                continue
            signal = df[ch].values
            features = extract_features_from_details(signal)
            writer = channel_files[ch]
            writer.write(f'S{subject:02d},G{game},' + ','.join(map(str, features)) + '\n')

# Close files
for f in channel_files.values():
    f.close()
